<a href="https://colab.research.google.com/github/anan5093/satellite-radar-flood-pipeline/blob/main/Radar_Flood_Mapping_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 40.0 MB/s eta 0:00:00


In [2]:
import ee

In [5]:
import geemap
ee.Authenticate()
PROJECT_ID = 'radar-flood-project'
ee.Initialize(project=PROJECT_ID)
print("Google Earth Engine successfully initialized")

Google Earth Engine successfully initialized


In [6]:
Map = geemap.Map()
Map
#

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [8]:
import ee
states = ee.FeatureCollection('FAO/GAUL/2015/level1')
indian_states = states.filter(ee.Filter.eq('ADM0_NAME', 'India'))
taret_states_list = ['Bihar', 'Assam', 'Uttar Pradesh', 'West Bengal']
flood_zones = indian_states.filter(ee.Filter.inList('ADM1_NAME', taret_states_list))
Map.addLayer(flood_zones, {}, 'Flood Zones')
macro_roi = flood_zones.geometry()
Map.centerObject(macro_roi, 6)
Map
print("Macro region established for given states")

Macro region established for given states


In [10]:
sar_collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
                    .filterBounds(macro_roi)
                    .filter(ee.Filter.eq('instrumentMode', 'IW'))
                    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')))

In [12]:
before_monsoon = sar_collection.filterDate('2024-04-01', '2024-05-15').mosaic().clip(macro_roi)
after_monsoon = sar_collection.filterDate('2024-07-01', '2024-08-31').mosaic().clip(macro_roi)
before_clean = before_monsoon.focal_median(30, 'circle', 'meters')
during_clean = after_monsoon.focal_median(30, 'circle', 'meters')
print("All-weather satellite radar composites generated acorss all target states")

All-weather satellite radar composites generated acorss all target states


In [14]:
import geemap
Map3 = geemap.Map()
Map3.setCenter(85.0, 25.0, 5)
vis_params = {
    'min': -25,
    'max': 0,
    'palette': ['blue', 'green', 'red']
}
Map3.addLayer(before_clean.select('VV'), vis_params, 'Dry Baseline (Pre-Monsoon)')
Map3.addLayer(during_clean.select('VV'), vis_params, 'Monsoon Peak (Flood-window)')
outline = ee.Image().paint(flood_zones, 0, 2)
Map3.addLayer(outline.updateMask(outline), {'palette': 'red'}, 'Target State Boundaries')
Map3

Map(center=[25.0, 85.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [15]:
monsoon_water = during_clean.select('VV').lt(-16)
permanent_water = before_clean.select('VV').lt(-16)

flooded_areas = monsoon_water.subtract(permanent_water).eq(1)

flood_mask = flooded_areas.updateMask(flooded_areas)
print("Automated flood detection engine compiled successfully.")

Automated flood detection engine compiled successfully.


In [18]:
area_image = flood_mask.multiply(ee.Image.pixelArea())

stats = area_image.reduceRegion(**{
    'reducer': ee.Reducer.sum().unweighted(),
    'geometry': macro_roi,
    'scale': 1000,  # Increased scale to 1000m to prevent timeout on large region
    'maxPixels': 1e10,
    'bestEffort': True
})

flooded_sq_km = ee.Number(stats.get('VV')).divide(1e6).round() # Corrected division for sq km conversion

print(f"Calculation complete! Total detected flooded area across target states: {flooded_sq_km.getInfo()} sq km")

Calculation complete! Total detected flooded area across target states: 3966 sq km


In [22]:
Map4 = geemap.Map()
Map4.setCenter(85.0, 25.0, 5)
Map4.addLayer(outline, {'palette': '555555'}, 'State Boundaries')
Map4.addLayer(flood_mask, {'palette': 'red'}, 'Automatically Detected Floods (Monsoon 2024)')
Map4

Map(center=[25.0, 85.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [31]:
population_collection = ee.ImageCollection('WorldPop/GP/100m/pop')
# Filter for a recent year (e.g., 2020) and select the 'population' band
india_pop = (population_collection.filter(ee.Filter.eq('country', 'IND')).filter(ee.Filter.eq('year', 2020)).select('population').first())

affected_population_zones = india_pop.updateMask(flood_mask)
total_affected_population = affected_population_zones.reduceRegion(**{
    'reducer': ee.Reducer.sum(),
    'geometry': macro_roi,
    'scale': 1000,
    'maxPixels': 1e10,
    'bestEffort': True
})
print(f"Total affected population in India: {total_affected_population.get('population').getInfo()}")

Total affected population in India: 28710.23662582513


In [32]:
import geemap
Map5 = geemap.Map()
Map5.setCenter(85.0, 25.0, 5)
Map5.addLayer(outline, {'palette': '333333'}, 'State Boundaries')
Map5.addLayer(flood_mask, {'palette': 'ADD8E6'}, 'Water Footprint')

pop_vis = {'min': 0, 'max': 10, 'palette': ['#FFFFB2', '#FECC5C', '#FD8D3C', '#E31A1C']}
Map5.addLayer(affected_population_zones, pop_vis, 'Human Impact Heatmap')
Map5

Map(center=[25.0, 85.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …